In [1]:
import pandas as pd
import numpy as np
import music21
import glob
import re

In [2]:
# From https://osf.io/wgz9t/
melody_names = pd.read_csv("./data/new_melody_numbers.csv")
human_responses = pd.read_csv("./data/exp2_data.csv")

In [3]:
filtered_responses = human_responses[
    human_responses['item'].str.startswith(tuple(melody_names['encoded'])) &
    (human_responses['MIDI'] != 0)
]

mask = filtered_responses['item'].str.contains('am')
filtered_responses.loc[mask, 'MIDI'] += 12
filtered_responses['item'] = filtered_responses['item'].str.replace(r'am?', '', regex=True)


/var/folders/36/5bbx7vj9617flhb60_frs0vw0000gn/T/ipykernel_48732/2114864029.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_responses['item'] = filtered_responses['item'].str.replace(r'am?', '', regex=True)


In [4]:
filtered_responses['adjusted_MIDI'] = filtered_responses['MIDI'] + filtered_responses['centsOff'] / 100
mean_midi = filtered_responses.groupby('item').agg(
    MIDI=('MIDI', 'mean'),
    MIDI_cents=('adjusted_MIDI', 'mean')
).reset_index()

mean_midi

/var/folders/36/5bbx7vj9617flhb60_frs0vw0000gn/T/ipykernel_48732/60874292.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_responses['adjusted_MIDI'] = filtered_responses['MIDI'] + filtered_responses['centsOff'] / 100


,item,MIDI,MIDI_cents
0,HC10,63.423077,63.345567
1,HC11,63.666667,63.767050
2,HC12,68.583333,68.596151
3,HC13,60.307692,60.358277
4,HC14,64.769231,64.728371
...,...,...,...
87,LC5,61.153846,61.088091
88,LC6,65.375000,65.385376
89,LC7,66.153846,66.137516
90,LC8,68.115385,68.059208


In [5]:
mean_midi.to_csv('exports/aggregate_human_responses.csv', index=False)
filtered_responses[['item', 'MIDI']].to_csv('exports/individual_responses.csv', index=False)

In [6]:
midi_files = sorted(glob.glob('data/midi/*.mid'))

melodies = {}

for filepath in midi_files:
    name = re.sub(r'\.mid$', '', filepath.split('/')[-1])
    score = music21.converter.parse(filepath)
    notes = []
    for element in score.flatten().notes:
        if isinstance(element, music21.note.Note):
            notes.append(float(element.pitch.midi))
        elif isinstance(element, music21.chord.Chord):
            notes.append(float(element.root().midi))
    melodies[name] = notes

sorted_cols = sorted(melodies.keys(), key=lambda name: (int(re.search(r'\d+', name).group()), name))

max_len = max(len(v) for v in melodies.values())
for k in melodies:
    melodies[k] += [None] * (max_len - len(melodies[k]))

df = pd.DataFrame(melodies)[sorted_cols]

In [7]:
pat = re.compile(r'([A-Za-z]+)(\d+)')

df.columns = [f"{pat.match(c).group(1)}{int(pat.match(c).group(2)):02d}" for c in df.columns]

melody_names['original_padded'] = melody_names['original'].apply(
    lambda x: f"{pat.match(x).group(1)}{int(pat.match(x).group(2)):02d}"
)

encoded_to_padded = dict(zip(melody_names['encoded'], melody_names['original_padded']))

mean_midi['item_padded'] = mean_midi['item'].map(encoded_to_padded)
human_means = mean_midi.set_index('item_padded')['MIDI']

for col in df.columns:
    last_idx = df[col].last_valid_index()
    if col in human_means.index:
        df.loc[last_idx + 1, col] = human_means[col]

df.columns = [f"{pat.match(c).group(1)}{int(pat.match(c).group(2))}" for c in df.columns]

df.to_csv('data/all_melodies_final_means.csv', index=False)